In [0]:
metadata_df = spark.table("raw_catalog.labs_connect.labsconnect_sttm")

In [0]:
fnd_catalog = "raw_catalog"
src_schema_etl_control = "etl_control"

In [0]:
dbutils.widgets.help()

dbutils.widgets provides utilities for working with notebook widgets. You can create
different types of widgets and get their bound value.

For more info about a method, use dbutils.widgets.help("methodName") .
 combobox(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a combobox input widget with a given name, default value and choices dropdown(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a dropdown input widget a with given name, default value and choices get(name: String): String -> Retrieves current value of an input widget getAll: Map -> Retrieves a mapping of all current values of the input widgets getArgument(name: String, optional: String): String -> (DEPRECATED) Equivalent to get multiselect(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a multiselect input widget with a given name, default value and choices remove(name: String): void -> Removes an input widget from the notebook removeAll: void -> Removes all widgets in the notebook text(name: String, defaultValue: String, label: String): void -> Creates a text input widget with a given name and default value

In [0]:
%pip install pyyaml databricks-labs-dqx
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("table_name", "STTM")
dbutils.widgets.text("operation_type", "")
dbutils.widgets.text("src_system", "labsconnect")
dbutils.widgets.text("job_id", "")
dbutils.widgets.text("volume_path", "")
dbutils.widgets.text("conform_catalog", "edb_conform_bu_c360_dev")
dbutils.widgets.text("metadata_table", "metadata_mapping_address_dim_sheet.csv")
dbutils.widgets.text("src_catalog_name", "edb_refined_bu_c360_dev_test")
dbutils.widgets.text("tgt_catalog_name", "edb_conform_bu_c360_dev_test")

In [0]:
import os
import json
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.rule import DQRowRule, DQDatasetRule
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.config import TableChecksStorageConfig
import pandas as pd
from dataclasses import dataclass
from typing import Optional
from typing import Dict, Optional, Tuple, List

# from dotenv import load_dotenv
# load_dotenv()

In [0]:
def get_param(name: str, default: str = "") -> str:
    try:
        value = dbutils.widgets.get(name)  # Databricks
    except Exception:
        print(name)
        value = os.getenv(name, default)  # Local/env fallback
    return (value or default).strip()


@dataclass
class PipelineConfig:
    table_name: str
    operation_type: str
    src_system: str
    job_id: str
    volume_path: str
    conform_catalog: str
    metadata_table: str
    src_catalog_name: str
    tgt_catalog_name: str

    @property
    def log_table(self) -> str:
        return f"{self.conform_catalog}.etl_control.pipeline_audit_log"

    @property
    def control_table(self) -> str:
        return f"{self.conform_catalog}.etl_control.job_run_log"

    @property
    def pipeline_name(self) -> str:
        return f"SCD2_{self.table_name}_Pipeline"


def load_config() -> PipelineConfig:
    return PipelineConfig(
        table_name=get_param("table_name"),
        operation_type=get_param("operation_type"),
        src_system=get_param("src_system", "LSC"),
        job_id=get_param("job_id"),
        volume_path=get_param("volume_path"),
        conform_catalog=get_param("conform_catalog", "edb_conform_bu_c360_dev"),
        metadata_table=get_param("metadata_table"),
        src_catalog_name=get_param("src_catalog_name"),
        tgt_catalog_name=get_param("tgt_catalog_name"),
    )

In [0]:
def exec_sql(step_name: str, sql: str):
    print(f"\n--- Executing: {step_name} ---")
    print(sql)
    try:
        result = spark.sql(
            " ".join([s for _, s in sql]) if isinstance(sql, list) else sql
        )
        return result
    except Exception as e:
        raise


def escape_sql_literal(value: Optional[str]) -> str:
    if value is None:
        return ""
    return str(value).replace("'", "''")


def log_event(
    config: PipelineConfig,
    level: str,
    step: str,
    message: str,
    details: Optional[dict] = None,
):
    try:
        import json

        msg = escape_sql_literal(message)
        det = escape_sql_literal(json.dumps(details) if details else None)

        sql = f"""
        INSERT INTO {config.log_table}
        (job_id, ts, level, pipeline, step, table_name, src_system, message, details)
        VALUES (
            '{config.job_id}',
            current_timestamp(),
            '{level}',
            '{config.pipeline_name}',
            '{step}',
            '{config.table_name}',
            '{config.src_system}',
            {f"'{msg}'" if msg else "NULL"},
            {f"'{det}'" if det else "NULL"}
        )
        """
        exec_sql(step, sql)
    except Exception as e:
        meta = {}
        # As a last resort, print – but do not raise to avoid masking original errors
        control_table_update_failed = generate_control_table_update(
            config, meta, "", "", "", "FAILED"
        )
        # exec_sql(step ,control_table_update_failed)
        print(f"LOGGING FAILED [{level}] {step}: {message} | {details} :: {e}")

In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
import uuid
import json
from typing import Optional, Dict, List
from datetime import datetime


def log_event(
    config: PipelineConfig,
    level: str,
    step: str,
    message: str,
    status: str = "RUNNING",
    details: Optional[dict] = None,
    rows_in: Optional[int] = None,
    rows_out: Optional[int] = None,
    rows_rejected: Optional[int] = None,
    files_transferred: Optional[int] = None,
    table_name: Optional[str] = None,
    source: Optional[str] = None,
    error_message: Optional[str] = None,
    error_stack_trace: Optional[str] = None,
    s3_uris: Optional[List[str]] = None,
    file_paths: Optional[List[str]] = None,
    dq_triage: Optional[dict] = None,
):
    """
    Log a step-level event to pipeline_audit_log.
    
    Args:
        config: Pipeline configuration
        level: Log level (INFO | WARN | ERROR)
        step: Step name
        message: Log message
        status: Step status (RUNNING | SUCCESS | FAILED | SKIPPED)
        details: Additional details as dict
        rows_in: Input row count
        rows_out: Output row count
        rows_rejected: Rejected row count
        files_transferred: Files transferred count
        table_name: Target table name (overrides config.table_name)
        source: Source detail (cdb | ects | labsconnect | table_name)
        error_message: Error message (for ERROR level)
        error_stack_trace: Stack trace (for ERROR level)
        s3_uris: List of S3/Volume URIs
        file_paths: List of file paths
        dq_triage: Data quality triage info as dict
    """
    try:
        # Generate unique audit_id
        audit_id = str(uuid.uuid4())
        
        # Get table_name from parameter or config
        tbl_name = table_name if table_name else getattr(config, 'table_name', None)
        
        # Get batch_id from config if available
        batch_id = getattr(config, 'batch_id', None)
        
        # Get source from parameter or config
        src = source if source else getattr(config, 'source', None)
        
        # Try to get Databricks context
        notebook_run_id = None
        cluster_id = None
        notebook_path = None
        try:
            from pyspark.dbutils import DBUtils
            from pyspark.sql import SparkSession
            spark = SparkSession.getActiveSession()
            if spark:
                dbutils = DBUtils(spark)
                ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
                notebook_run_id = ctx.notebookId().get()
                cluster_id = ctx.clusterId().get()
                notebook_path = ctx.notebookPath().get()
        except:
            pass
        
        # Escape and prepare values
        msg = escape_sql_literal(message)
        det = escape_sql_literal(json.dumps(details) if details else "")
        err_msg = escape_sql_literal(error_message) if error_message else None
        err_stack = escape_sql_literal(error_stack_trace) if error_stack_trace else None
        dq_json = escape_sql_literal(json.dumps(dq_triage) if dq_triage else "")
        
        # Handle arrays
        s3_uris_sql = "NULL"
        if s3_uris:
            escaped_uris = [f"'{escape_sql_literal(uri)}'" for uri in s3_uris]
            s3_uris_sql = f"ARRAY({', '.join(escaped_uris)})"
        
        file_paths_sql = "NULL"
        if file_paths:
            escaped_paths = [f"'{escape_sql_literal(path)}'" for path in file_paths]
            file_paths_sql = f"ARRAY({', '.join(escaped_paths)})"
        
        # Handle NULL values
        rows_in_sql = "NULL" if rows_in is None else str(rows_in)
        rows_out_sql = "NULL" if rows_out is None else str(rows_out)
        rows_rejected_sql = "NULL" if rows_rejected is None else str(rows_rejected)
        files_transferred_sql = "NULL" if files_transferred is None else str(files_transferred)
        
        # Build INSERT statement
        sql = f"""
        INSERT INTO {config.log_table}
        (
            audit_id,
            job_id,
            pipeline,
            step,
            src_system,
            source,
            status,
            level,
            message,
            step_ts,
            rows_in,
            rows_out,
            rows_rejected,
            records_written,
            files_transferred,
            dq_triage,
            table_name,
            s3_uris,
            file_paths,
            error_message,
            error_stack_trace,
            details,
            notebook_run_id,
            cluster_id,
            notebook_path
        )
        VALUES (
            '{audit_id}',
            '{config.job_id}',
            '{escape_sql_literal(config.pipeline_name)}',
            '{escape_sql_literal(step)}',
            '{escape_sql_literal(config.src_system)}',
            {f"'{escape_sql_literal(src)}'" if src else "NULL"},
            '{status}',
            '{level}',
            '{msg}',
            current_timestamp(),
            {rows_in_sql},
            {rows_out_sql},
            {rows_rejected_sql},
            {rows_out_sql},
            {files_transferred_sql},
            {f"'{dq_json}'" if dq_triage else "NULL"},
            {f"'{escape_sql_literal(tbl_name)}'" if tbl_name else "NULL"},
            {s3_uris_sql},
            {file_paths_sql},
            {f"'{err_msg}'" if err_msg else "NULL"},
            {f"'{err_stack}'" if err_stack else "NULL"},
            {f"'{det}'" if details else "NULL"},
            {f"'{escape_sql_literal(notebook_run_id)}'" if notebook_run_id else "NULL"},
            {f"'{escape_sql_literal(cluster_id)}'" if cluster_id else "NULL"},
            {f"'{escape_sql_literal(notebook_path)}'" if notebook_path else "NULL"}
        )
        """
        
        exec_sql(step, sql)
        
    except Exception as e:
        # Fallback logging - print to console
        print(f"LOGGING FAILED [{level}] {step}: {message} | {details} :: {e}")


def escape_sql_literal(value: str) -> str:
    """Escape SQL string literals."""
    if value is None:
        return ""
    return str(value).replace("'", "''").replace("\\", "\\\\")


# Code Generated by Sidekick is for learning and experimentation purposes only.
def log_info(config: PipelineConfig, step: str, message: str, **kwargs):
    """Log an INFO level event."""
    log_event(config, "INFO", step, message, status="RUNNING", **kwargs)


def log_success(config: PipelineConfig, step: str, message: str, **kwargs):
    """Log a SUCCESS event."""
    log_event(config, "INFO", step, message, status="SUCCESS", **kwargs)


def log_warning(config: PipelineConfig, step: str, message: str, **kwargs):
    """Log a WARNING event."""
    log_event(config, "WARN", step, message, status="RUNNING", **kwargs)


def log_error(config: PipelineConfig, step: str, message: str, error: Exception = None, **kwargs):
    """Log an ERROR event with stack trace."""
    import traceback
    
    error_message = str(error) if error else message
    error_stack_trace = traceback.format_exc() if error else None
    
    log_event(
        config, 
        "ERROR", 
        step, 
        message, 
        status="FAILED",
        error_message=error_message,
        error_stack_trace=error_stack_trace,
        **kwargs
    )


def log_step_metrics(
    config: PipelineConfig,
    step: str,
    message: str,
    rows_in: int,
    rows_out: int,
    rows_rejected: int = 0,
    **kwargs
):
    """Log step completion with metrics."""
    log_event(
        config,
        "INFO",
        step,
        message,
        status="SUCCESS",
        rows_in=rows_in,
        rows_out=rows_out,
        rows_rejected=rows_rejected,
        **kwargs
    )



In [0]:
def validate_required_columns(pdf: pd.DataFrame):
    missing = [col for col in REQUIRED_COLUMNS if col not in pdf.columns]
    if missing:
        log_event("WARN", "Validation", f"Missing required columns: {missing}")
        raise ValueError(f"Missing required metadata columns: {missing}")

In [0]:
from typing import Dict, Optional


def control_lookup_sql(meta: Dict[str, str], config: PipelineConfig) -> str:
    """
    Get last successful run timestamp for incremental watermarking.
    
    Args:
        meta: Metadata dict with tgt_table, tgt_schema, src_system
        config: Pipeline configuration with control_table
    
    Returns:
        SQL query string
    """
    return (
        f"SELECT COALESCE(MAX(lst_run_ts), CAST('1900-01-01 00:00:00' AS TIMESTAMP)) "
        f"FROM {config.control_table} "
        f"WHERE sts = 'SUCCESS' "
        f"AND tbl_nm = '{meta['tgt_table']}' "
        f"AND schm_nm = '{meta['tgt_schema']}' "
        f"AND src_system = '{meta['src_system']}'"
    )


def generate_control_table_insert(
    control_table: str,
    pipeline_name: str,
    tbl_nm: str,
    schm_nm: str,
    src_system: str,
    job_id: str,
    batch_id: Optional[str] = None,
) -> str:
    """
    Generate MERGE statement to initialize job run record.
    
    Args:
        control_table: Control table name (job_run_log)
        pipeline_name: Pipeline name
        tbl_nm: Target table name
        schm_nm: Target schema name
        src_system: Source system identifier
        job_id: Unique job identifier (UUID)
        batch_id: Optional batch identifier
    
    Returns:
        MERGE SQL statement
    """
    
    # Try to get Databricks context
    try:
        from pyspark.dbutils import DBUtils
        from pyspark.sql import SparkSession
        spark = SparkSession.getActiveSession()
        if spark:
            dbutils = DBUtils(spark)
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            notebook_run_id = f"'{escape_sql_literal(ctx.notebookId().get())}'"
            cluster_id = f"'{escape_sql_literal(ctx.clusterId().get())}'"
            notebook_path = f"'{escape_sql_literal(ctx.notebookPath().get())}'"
        else:
            notebook_run_id = "NULL"
            cluster_id = "NULL"
            notebook_path = "NULL"
    except:
        notebook_run_id = "NULL"
        cluster_id = "NULL"
        notebook_path = "NULL"
    
    return f"""
    MERGE INTO {control_table} AS t
    USING (
        SELECT
            '{escape_sql_literal(job_id)}' AS job_id,
            '{escape_sql_literal(tbl_nm)}' AS tbl_nm,
            '{escape_sql_literal(pipeline_name)}' AS ppln_nm,
            '{escape_sql_literal(schm_nm)}' AS schm_nm,
            '{escape_sql_literal(src_system)}' AS src_system,
            'RUNNING' AS sts,
            NULL AS run_nts,
            current_timestamp() AS crtd_at,
            current_timestamp() AS updt_at,
            current_timestamp() AS lst_run_ts,
            NULL AS jb_end_ts,
            '' AS merge_query,
            '' AS insert_query,
            '' AS ddl_query,
            CAST(NULL AS BIGINT) AS src_count,
            CAST(NULL AS BIGINT) AS tgt_count,
            CAST(NULL AS BIGINT) AS reject_count,
            CAST(NULL AS INT) AS files_transferred,
            {notebook_run_id} AS notebook_run_id,
            {cluster_id} AS cluster_id,
            {notebook_path} AS notebook_path
    ) s
    ON t.job_id = s.job_id
       AND t.tbl_nm = s.tbl_nm
       AND t.schm_nm = s.schm_nm
       AND t.src_system = s.src_system
    WHEN MATCHED AND t.sts IN ('RUNNING', 'FAILED') THEN UPDATE SET
        t.ppln_nm = s.ppln_nm,
        t.sts = s.sts,
        t.run_nts = s.run_nts,
        t.updt_at = s.updt_at,
        t.lst_run_ts = s.lst_run_ts,
        t.notebook_run_id = s.notebook_run_id,
        t.cluster_id = s.cluster_id,
        t.notebook_path = s.notebook_path
    WHEN NOT MATCHED THEN INSERT (
        job_id,
        ppln_nm,
        lst_run_ts,
        sts,
        run_nts,
        crtd_at,
        updt_at,
        tbl_nm,
        schm_nm,
        merge_query,
        insert_query,
        ddl_query,
        src_system,
        jb_end_ts,
        src_count,
        tgt_count,
        reject_count,
        files_transferred,
        notebook_run_id,
        cluster_id,
        notebook_path
    ) VALUES (
        s.job_id,
        s.ppln_nm,
        s.lst_run_ts,
        s.sts,
        s.run_nts,
        s.crtd_at,
        s.updt_at,
        s.tbl_nm,
        s.schm_nm,
        s.merge_query,
        s.insert_query,
        s.ddl_query,
        s.src_system,
        s.jb_end_ts,
        s.src_count,
        s.tgt_count,
        s.reject_count,
        s.files_transferred,
        s.notebook_run_id,
        s.cluster_id,
        s.notebook_path
    )
    """


def generate_control_table_update(
    config: PipelineConfig,
    meta: Dict[str, str],
    merge_query: str,
    insert_query: str,
    ddl_query: str,
    status: str,
    src_count: Optional[int] = None,
    tgt_count: Optional[int] = None,
    reject_count: Optional[int] = None,
    files_transferred: Optional[int] = None,
    notes: Optional[str] = None,
) -> str:
    """
    Generate MERGE statement to update job run with final results.
    
    Args:
        config: Pipeline configuration with control_table, job_id, pipeline_name
        meta: Metadata dict with tgt_table, tgt_schema, src_system
        merge_query: Merge query executed
        insert_query: Insert query executed
        ddl_query: DDL query executed
        status: Job status (SUCCESS or FAILED)
        src_count: Source record count
        tgt_count: Target record count
        reject_count: Rejected record count
        files_transferred: Number of files transferred
        notes: Additional notes or error message
    
    Returns:
        MERGE SQL statement
    """
    # Normalize status to uppercase
    success_fail_val = "SUCCESS" if status.upper() == "SUCCESS" else "FAILED"
    
    # Escape queries - replace triple quotes and escape single quotes
    def escape_query(query: str) -> str:
        if not query:
            return ""
        return query.replace('"""', "'").replace("'", "''")
    
    merge_query_escaped = escape_query(merge_query)
    insert_query_escaped = escape_query(insert_query)
    ddl_query_escaped = escape_query(ddl_query)
    notes_escaped = escape_query(notes) if notes else ""
    
    # Handle NULL values
    src_count_sql = "NULL" if src_count is None else str(src_count)
    tgt_count_sql = "NULL" if tgt_count is None else str(tgt_count)
    reject_count_sql = "NULL" if reject_count is None else str(reject_count)
    files_transferred_sql = "NULL" if files_transferred is None else str(files_transferred)
    
    # Get metadata
    tgt_table = meta.get("tgt_table", "")
    tgt_schema = meta.get("tgt_schema", "")
    src_system = meta.get("src_system", "")
    
    return f"""
    MERGE INTO {config.control_table} AS t
    USING (
        SELECT
            '{escape_sql_literal(config.job_id)}' AS job_id,
            '{escape_sql_literal(tgt_table)}' AS tbl_nm,
            '{escape_sql_literal(tgt_schema)}' AS schm_nm,
            '{escape_sql_literal(src_system)}' AS src_system,
            '{escape_sql_literal(config.pipeline_name)}' AS ppln_nm,
            '{success_fail_val}' AS sts,
            '{notes_escaped}' AS run_nts,
            current_timestamp() AS updt_at,
            current_timestamp() AS jb_end_ts,
            '{merge_query_escaped}' AS merge_query,
            '{insert_query_escaped}' AS insert_query,
            '{ddl_query_escaped}' AS ddl_query,
            {src_count_sql} AS src_count,
            {tgt_count_sql} AS tgt_count,
            {reject_count_sql} AS reject_count,
            {files_transferred_sql} AS files_transferred
    ) s
    ON t.job_id = s.job_id
       AND t.tbl_nm = s.tbl_nm
       AND t.schm_nm = s.schm_nm
       AND t.src_system = s.src_system
    WHEN MATCHED AND t.sts IN ('RUNNING', 'FAILED') THEN UPDATE SET
        t.ppln_nm = s.ppln_nm,
        t.sts = s.sts,
        t.run_nts = s.run_nts,
        t.updt_at = s.updt_at,
        t.jb_end_ts = s.jb_end_ts,
        t.merge_query = s.merge_query,
        t.insert_query = s.insert_query,
        t.ddl_query = s.ddl_query,
        t.src_count = s.src_count,
        t.tgt_count = s.tgt_count,
        t.reject_count = s.reject_count,
        t.files_transferred = s.files_transferred
    """


def escape_sql_literal(value: str) -> str:
    """
    Escape SQL string literals.
    
    Args:
        value: String value to escape
    
    Returns:
        Escaped string safe for SQL
    """
    if value is None:
        return ""
    return str(value).replace("'", "''").replace("\\", "\\\\")


In [0]:
def get_dq_path(volume_path: str) -> str:
    local_source_dir = os.getcwd()
    path_parts = local_source_dir.split("/")
    if "files" in path_parts:
        path_until_files = "/".join(path_parts[: path_parts.index("files") + 1])
        return os.path.join(path_until_files, "resources/dq_rules")
    return f"{volume_path.rstrip('/')}/DQ_metadata"


def standardize_column_names(pdf, column_map):
    col_map_lower = {k.lower().strip(): v for k, v in column_map.items()}
    new_cols = [col_map_lower.get(str(c).lower().strip(), c) for c in pdf.columns]
    pdf = pdf.toDF(*new_cols)
    return pdf


def validate_required_columns(pdf: pd.DataFrame):
    missing = [col for col in REQUIRED_COLUMNS if col not in pdf.columns]
    if missing:
        log_event("WARN", "Validation", f"Missing required columns: {missing}")
        raise ValueError(f"Missing required metadata columns: {missing}")

In [0]:
def load_metadata_from_table(config: PipelineConfig, COLUMN_NAME_MAP) -> pd.DataFrame:
    if not config.metadata_table:
        raise ValueError("metadata_table widget is required")
    # try:
    #     metadata_pdf = spark.table(config.metadata_table).toPandas()
    # except Exception as e:
    #     log_event("ERROR", "Databricks table read failed", f"Table load failed for {config.metadata_table}: {e}")
    #     raise

    try:
        # .option("multiline", "false")
        file_name = "metadata_mapping_address_dim_sheet.csv"
        file_name = "/Volumes/raw_catalog/etl_control/test/labsconnect_sttm_sheet.csv"
        # metadata_df = spark.read.format("csv").option("multiline", "false").option("header", True).option("inferSchema",
        #                                                                                                True).option(
        #     "mergeSchema", "true").load(file_name)
        metadata_df = spark.table("raw_catalog.labs_connect.labsconnect_sttm")

        ### Start executing dynamic read
    except Exception as e:
        log_event(
            "ERROR",
            "STTM file Read failed",
            f"File source load failed for {config.volume_path}: {e}",
        )
        raise

    if (
        "Source Field Name" in metadata_df.columns
        and "Source Field Name (Table order)" in metadata_df.columns
    ):
        metadata_df = metadata_df.drop("Source Field Name")

    # creating same columns from config by overriding
    column_config_map = {
        "source catalog name": config.src_catalog_name,
        "target table catalog": config.tgt_catalog_name,
    }

    from pyspark.sql.functions import lit

    for col, value in column_config_map.items():
        metadata_df = metadata_df.withColumn(col, lit(value))

    metadata_df = standardize_column_names(metadata_df, COLUMN_NAME_MAP)

    if "Target Table/File Name" not in metadata_df.columns:
        raise ValueError("Metadata table must contain 'Target Table/File Name'")

    metadata_pdf = metadata_df.toPandas()
    metadata_pdf = metadata_pdf[
        metadata_pdf["Target Table/File Name"].astype(str).str.strip()
        == config.table_name
    ].copy()

    if metadata_pdf.empty:
        raise ValueError(f"No metadata found for table_name={config.table_name}")

    if "Source System" not in metadata_pdf.columns:
        metadata_pdf["Source System"] = config.src_system
    else:
        metadata_pdf["Source System"] = metadata_pdf["Source System"].fillna("LSC")

    metadata_pdf = metadata_pdf[
        metadata_pdf["Source System"].astype(str).str.strip().str.lower()
        == config.src_system.lower()
    ].copy()

    if metadata_pdf.empty:
        raise ValueError(
            f"No metadata found for table_name={config.table_name} and src_system={config.src_system}"
        )

    metadata_pdf = metadata_pdf.fillna("")
    validate_required_columns(metadata_pdf)
    return metadata_pdf


def get_filter_list(sttm_df: pd.DataFrame) -> List[str]:
    if "filter" not in sttm_df.columns:
        return []
    series = sttm_df["filter"].fillna("").astype(str).str.strip()
    return [v for v in series.tolist() if v]


def build_incremental_where_clause(
    sttm_df: pd.DataFrame,
    meta: Dict[str, str],
    config: PipelineConfig,
    base_alias: str,
    source_alias_fields: List[str],
    source_fields_clause: str,
) -> str:
    lookup_sql = control_lookup_sql(meta, config)
    filter_list = get_filter_list(sttm_df)
    filter_string = " AND ".join(filter_list)

    if (
        not config.table_name.startswith("lnd")
        and config.src_system.upper() not in ("PPH", "EPH")
        and len(source_alias_fields) > 1
    ):
        greatest_expr = ", ".join([f"{a}.lst_updt_ts" for a in source_alias_fields])
        where_clause = f"GREATEST({greatest_expr}) >= ({lookup_sql})"
    else:
        where_clause = f"{base_alias}.lst_updt_ts >= ({lookup_sql})"

    if config.src_system.upper() == "PPH":
        where_clause = f"COALESCE({base_alias}.updatedtime_format, current_timestamp()) >= ({lookup_sql})"

    if config.src_system.upper() == "EPH" and "updateddate" in source_fields_clause:
        where_clause = (
            f"COALESCE({base_alias}.updateddate, current_timestamp()) >= ({lookup_sql})"
        )

    if config.src_system.upper() == "EPH" and "src_updtd_dt" in source_fields_clause:
        where_clause = f"COALESCE({base_alias}.src_updtd_dt, current_timestamp()) >= ({lookup_sql})"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "LSC"
        and "LastModifiedDate__c" in source_fields_clause
    ):
        where_clause = f"{base_alias}.LastModifiedDate__c >= ({lookup_sql})"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "LSC"
        and "Last_Modified_Date__c" in source_fields_clause
    ):
        where_clause = f"{base_alias}.Last_Modified_Date__c >= (({lookup_sql}) - INTERVAL 60 MINUTES)"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "LSC"
        and "ssot__LastModifiedDate__c" in source_fields_clause
    ):
        where_clause = f"{base_alias}.ssot__LastModifiedDate__c >= (({lookup_sql}) - INTERVAL 60 MINUTES)"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "CAPGEMINI"
        and "INSERT_TIMESTAMP" in source_fields_clause
    ):
        where_clause = ""

    if config.table_name.startswith("lnd") and config.src_system.upper() == "CTI":
        where_clause = (
            f"to_timestamp({base_alias}.last_update_timestamp) >= ({lookup_sql})"
        )

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "CAPGEMINI"
        and "INSERT_TIMESTAMP" not in source_fields_clause
    ):
        where_clause = ""

    if config.table_name.startswith("lnd") and config.src_system.upper() == "PAYSIGN":
        where_clause = ""

    if where_clause and filter_string:
        return f"{where_clause} AND {filter_string}"

    if not where_clause and filter_string:
        return filter_string

    return where_clause


def select_incremental_ts_column(
    config: PipelineConfig, source_fields_clause: str
) -> str:
    incremental_ts_col = "lst_updt_ts"

    if config.src_system.upper() == "PPH":
        incremental_ts_col = "updatedtime_format"

    if config.src_system.upper() == "EPH" and "updateddate" in source_fields_clause:
        incremental_ts_col = "updateddate"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "LSC"
        and "LastModifiedDate__c" in source_fields_clause
    ):
        incremental_ts_col = "LastModifiedDate__c"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "LSC"
        and "Last_Modified_Date__c" in source_fields_clause
    ):
        incremental_ts_col = "Last_Modified_Date__c"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "LSC"
        and "ssot__LastModifiedDate__c" in source_fields_clause
    ):
        incremental_ts_col = "ssot__LastModifiedDate__c"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "CAPGEMINI"
        and "INSERT_TIMESTAMP" in source_fields_clause
    ):
        incremental_ts_col = ""

    if config.table_name.startswith("lnd") and config.src_system.upper() == "CTI":
        incremental_ts_col = "last_update_timestamp"

    if (
        config.table_name.startswith("lnd")
        and config.src_system.upper() == "CAPGEMINI"
        and "INSERT_TIMESTAMP" not in source_fields_clause
    ):
        incremental_ts_col = ""

    if config.table_name.startswith("lnd") and config.src_system.upper() == "PAYSIGN":
        incremental_ts_col = ""

    return incremental_ts_col


def reorder_etl_job_id_last(pdf: pd.DataFrame) -> pd.DataFrame:
    if "Target Field Name" not in pdf.columns:
        return pdf
    etl_df = pdf[
        pdf["Target Field Name"].astype(str).str.strip() == "etl_job_id"
    ].copy()
    non_etl_df = pdf[
        pdf["Target Field Name"].astype(str).str.strip() != "etl_job_id"
    ].copy()
    if etl_df.empty:
        return non_etl_df
    return pd.concat([non_etl_df, etl_df], ignore_index=True)


def generate_ddl(sttm_df: pd.DataFrame, operation_flag: str) -> Tuple[str, List[str]]:
    if operation_flag.lower() != "create_table":
        return "", []

    ddl_statements = []
    masking_commands = []
    partition_cols = []
    catalog = ""
    schema = ""
    table_name = ""

    for _, row in sttm_df.iterrows():
        catalog = str(row["Target Table Catalog"]).strip()
        schema = str(row["Target Table Schema"]).strip()
        table_name = str(row["Target Table/File Name"]).strip()
        field_name = str(row["Target Field Name"]).strip()
        data_type = str(row["Target Data Type With Length"]).strip()
        primary_key = str(row["Primary Key"]).strip()
        column_description = escape_sql_literal(
            str(row.get("Target Column Description", "")).strip()
        )
        transformation = str(row.get("Transformation", "")).strip()
        partition_by = str(row.get("Partition By", "")).strip()
        masking_flag = str(row.get("Masking", "")).strip().upper()

        if partition_by:
            partition_cols.append(partition_by)

        mask_clause = ""
        if masking_flag == "Y":
            masking_func = f"{catalog}.{schema}.mask_{field_name}"
            # check
            masking_commands.append(
                f"CREATE OR REPLACE FUNCTION {masking_func} ({field_name} STRING) "
                f"RETURN CASE WHEN is_member('dbx_bu_c360_data_engineer') "
                f"THEN {field_name} ELSE '*********' END"
            )
            mask_clause = f" MASK {masking_func}"

        if transformation == "Autogenerated":
            ddl_details = (
                f"{field_name} {data_type} NOT NULL "
                f"GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) "
                f"COMMENT '{column_description}'"
            )
        elif primary_key.upper() == "PK":
            ddl_details = f"{field_name} {data_type}{mask_clause} NOT NULL COMMENT '{column_description}'"
        else:
            ddl_details = (
                f"{field_name} {data_type}{mask_clause} COMMENT '{column_description}'"
            )

        ddl_statements.append(ddl_details)

    pk_fields = [
        str(row["Target Field Name"]).strip()
        for _, row in sttm_df.iterrows()
        if str(row["Primary Key"]).strip().upper() == "PK"
    ]
    pk_clause = ""
    if pk_fields:
        pk_cols = ", ".join([f"`{c}`" for c in pk_fields])
        pk_clause = f",\n        CONSTRAINT `{table_name}_pk` PRIMARY KEY ({pk_cols})"

    partition_clause = ""
    if partition_cols:
        unique_partition_cols = ", ".join(sorted(set([c for c in partition_cols if c])))
        if unique_partition_cols:
            partition_clause = f"\n    PARTITIONED BY ({unique_partition_cols})"

    ddl = f"""
    CREATE OR REPLACE TABLE {catalog}.{schema}.{table_name} (
        {','.join(ddl_statements)}{pk_clause}
    ){partition_clause};
    """
    return ddl.strip(), masking_commands


def generate_source_subquery(
    select_exprs: List[str],
    base_table: str,
    join_clauses: List[str],
    where_clause: Optional[str],
    config: PipelineConfig,
) -> str:
    select_clause = ",\n            ".join(select_exprs)
    join_str = "".join([f"\n        {join}" for join in join_clauses])
    where_str = f"\n        WHERE {where_clause}" if where_clause else ""

    if config.src_system.lower() in ("eph", "pph"):
        distinct_word = "DISTINCT"
    elif config.table_name.startswith("lnd"):
        distinct_word = ""
    else:
        distinct_word = "DISTINCT"

    return f"""
    (
        SELECT {distinct_word}
            {select_clause}
        FROM {base_table}{join_str}{where_str}
    )
    """.strip()


def build_mapping_from_sttm(
    sttm_df: pd.DataFrame,
    clause_type: str,
    meta: Dict[str, str],
    config: PipelineConfig,
) -> Tuple[List[str], List[str], List[str], List[str], str, List[str], str, List[str]]:
    select_exprs = []
    tgt_fields = []
    business_keys = []
    src_business_keys = []
    compare_fields = []
    join_clauses = set()
    base_table = None

    audit_field_list = list(AUDIT_FIELD_DEFAULTS.keys()) + [
        "opn_flg",
        "etl_job_id",
        "ext_fld_1",
    ]

    source_fields = [
        str(row["Source Field Name"]).strip() for _, row in sttm_df.iterrows()
    ]
    source_fields_clause = ", ".join(source_fields)
    base_alias = meta["src_alias"]

    for _, row in sttm_df.iterrows():
        src_catalog = str(row["Source Catalog Name"]).strip()
        src_schema = str(row["Source Schema Name"]).strip()
        src_table = str(row["Source Table/File Name"]).strip()
        src_alias = str(row["Source Table Alias"]).strip()

        if not base_table and src_table and src_alias:
            base_table = f"{src_catalog}.{src_schema}.{src_table} AS {src_alias}"

        join_text = str(row.get("Joins", "")).strip()
        if join_text:
            join_clauses.add(join_text)

    for _, row in sttm_df.iterrows():
        tgt = str(row["Target Field Name"]).strip()
        src_alias = str(row["Source Table Alias"]).strip()
        src_fld = str(row["Source Field Name"]).strip()
        mapping_type = str(row["Direct vs Indirect Field Mapping"]).strip()
        pk_flag = str(row["Primary Key"]).strip().upper()
        transformation = str(row["Transformation"]).strip()

        if pk_flag == "PK":
            business_keys.append(tgt)
            src_business_keys.append(f"{src_alias}.{src_fld}")

        if pk_flag != "PK" and tgt and tgt not in audit_field_list:
            compare_fields.append(tgt)

        expr = None

        if src_fld and ";" in src_fld and not transformation:
            fields = [f.strip() for f in src_fld.split(";") if f.strip()]
            if len(fields) > 1:
                expr = f"CONCAT({', '.join([f'{src_alias}.{f}' for f in fields])}) AS {tgt}"
            elif len(fields) == 1:
                expr = f"{src_alias}.{fields[0]} AS {tgt}"

        if expr is None:
            if transformation:
                expr = f"{transformation} AS {tgt}"
            elif mapping_type == "Direct":
                if src_fld:
                    expr = f"{src_alias}.{src_fld} AS {tgt}"
                else:
                    expr = AUDIT_FIELD_DEFAULTS.get(tgt.lower(), f"NULL AS {tgt}")
            elif mapping_type == "Derived":
                expr = AUDIT_FIELD_DEFAULTS.get(tgt.lower(), f"NULL AS {tgt}")
            else:
                expr = AUDIT_FIELD_DEFAULTS.get(tgt.lower(), f"NULL AS {tgt}")

        if tgt and tgt.lower() not in ("opn_flg", "ext_fld_1", "etl_job_id"):
            select_exprs.append(expr)
            tgt_fields.append(tgt)

    if clause_type == "merge_insert":
        select_exprs.extend(
            [
                "'I' AS opn_flg",
                "NULL AS ext_fld_1",
                f"'{config.job_id}' AS etl_job_id",
            ]
        )
    elif clause_type == "late_arriving_insert":
        select_exprs.extend(
            [
                "'U' AS opn_flg",
                "NULL AS ext_fld_1",
                f"'{config.job_id}' AS etl_job_id",
            ]
        )
    else:
        select_exprs.extend(
            [
                "NULL AS opn_flg",
                "NULL AS ext_fld_1",
                f"'{config.job_id}' AS etl_job_id",
            ]
        )

    tgt_fields.extend(["opn_flg", "ext_fld_1", "etl_job_id"])

    if config.table_name.startswith("lnd") or config.src_system.lower() in (
        "eph",
        "pph",
    ):
        incremental_ts_col = select_incremental_ts_column(config, source_fields_clause)

        if not src_business_keys:
            raise ValueError("Primary key metadata is required for RN logic")

        if config.src_system.lower() == "paysign" or incremental_ts_col == "":
            order_expr = ", ".join(src_business_keys)
        else:
            order_expr = f"{base_alias}.{incremental_ts_col}"

        rn_expr = (
            f"ROW_NUMBER() OVER (PARTITION BY {', '.join(src_business_keys)} "
            f"ORDER BY {order_expr} DESC) AS RN"
        )
        select_exprs.append(rn_expr)

    source_alias_fields = sorted(
        list(
            set(
                str(row["Source Table Alias"]).strip()
                for _, row in sttm_df.iterrows()
                if str(row["Source Table Alias"]).strip()
            )
        )
    )

    return (
        tgt_fields,
        select_exprs,
        business_keys,
        compare_fields,
        base_table,
        sorted(list(join_clauses)),
        source_fields_clause,
        source_alias_fields,
    )


def get_default(dtype: str) -> str:
    dtype = str(dtype).lower()
    if dtype.startswith(("varchar", "char", "string")):
        return "''"
    if dtype.startswith(("double", "float", "decimal")):
        return "0.0"
    if dtype.startswith(("int", "bigint", "long", "short", "byte")):
        return "0"
    if dtype.startswith("date"):
        return "CAST('1900-01-01' AS DATE)"
    if dtype.startswith(("timestamp", "timestamp_ltz", "timestamp_ntz")):
        return "CAST('1900-01-01 00:00:00' AS TIMESTAMP)"
    if dtype.startswith(("bool", "boolean")):
        return "FALSE"
    return "NULL"


def build_when_matched_condition(
    sttm_df: pd.DataFrame, compare_fields: List[str], target_table: str
) -> str:
    source_field_map = {}
    source_data_type = {}
    target_data_type = {}

    for _, row in sttm_df.iterrows():
        target_field = str(row["Target Field Name"]).strip()
        source_field = str(row["Source Field Name"]).strip()
        src_dtype = (
            str(row["Data Type with length"]).strip()
            if "Data Type with length" in row
            else ""
        )
        tgt_dtype = str(row["Target Data Type With Length"]).strip()

        if (
            target_field
            and source_field
            and (
                target_field in compare_fields
                or (
                    target_table.endswith(
                        f"hipaa_quarantine_{target_table.split('.')[-1]}"
                    )
                    and target_field == "etl_job_id"
                )
            )
        ):
            source_field_map[target_field] = source_field
            source_data_type[target_field] = src_dtype
            target_data_type[target_field] = tgt_dtype

    compare_clauses = []
    for target_field in source_field_map.keys():
        src_default = get_default(source_data_type[target_field])
        tgt_default = get_default(target_data_type[target_field])

        if src_default != tgt_default:
            if tgt_default == "''":
                final_default = src_default
            elif src_default == "''":
                final_default = tgt_default
            else:
                final_default = tgt_default
        else:
            final_default = tgt_default

        compare_clauses.append(
            f"COALESCE(tgt.{target_field}, {final_default}) <> COALESCE(src.{target_field}, {final_default})"
        )

    return " OR\n        ".join(compare_clauses) if compare_clauses else "1 = 0"


def generate_merge_statement_from_sttm(
    sttm_df: pd.DataFrame, meta: Dict[str, str], config: PipelineConfig
) -> str:
    (
        tgt_fields,
        select_exprs,
        business_keys,
        compare_fields,
        base_table,
        join_clauses,
        source_fields_clause,
        source_alias_fields,
    ) = build_mapping_from_sttm(sttm_df, "merge_insert", meta, config)

    if not business_keys:
        raise ValueError("At least one PK is required in metadata")

    target_table = meta["tgt_full_name"]
    target_alias = meta["tgt_alias"]
    base_alias = meta["src_alias"]

    on_conditions = [f"{target_alias}.{k} <=> src.{k}" for k in business_keys]
    on_conditions.append(f"{target_alias}.is_actv = TRUE")

    if target_table.endswith(f"hipaa_quarantine_{config.table_name}"):
        on_conditions.append(f"{target_alias}.etl_job_id <=> src.etl_job_id")

    where_clause = build_incremental_where_clause(
        sttm_df=sttm_df,
        meta=meta,
        config=config,
        base_alias=base_alias,
        source_alias_fields=source_alias_fields,
        source_fields_clause=source_fields_clause,
    )

    source_query = generate_source_subquery(
        select_exprs, base_table, join_clauses, where_clause, config
    )

    if config.table_name.startswith("lnd") or config.src_system.lower() in (
        "eph",
        "pph",
    ):
        source_query = f"(SELECT * FROM {source_query} WHERE RN = 1)"

    if config.operation_type == "data_load_with_hipaa" and not target_table.endswith(
        f"hipaa_quarantine_{config.table_name}"
    ):
        source_query = (
            f"(SELECT * FROM {source_query} WHERE consent_dispense_active_flag = 1)"
        )

    if config.operation_type == "data_load_with_hipaa" and target_table.endswith(
        f"hipaa_quarantine_{config.table_name}"
    ):
        source_query = (
            f"(SELECT * FROM {source_query} WHERE consent_dispense_active_flag = 0)"
        )

    when_matched_condition = build_when_matched_condition(
        sttm_df, compare_fields, target_table
    )

    update_clause = ",\n        ".join(
        [
            "eff_end_dt = CURRENT_DATE()",
            "is_actv = FALSE",
            "lst_updt_ts = CURRENT_TIMESTAMP()",
            "updt_dt = CURRENT_DATE()",
            "updt_by = 'SYSTEM'",
            f"etl_job_id = '{config.job_id}'",
        ]
    )

    insert_fields_clause = ",\n        ".join(tgt_fields)
    insert_values_clause = ",\n        ".join([f"src.{f}" for f in tgt_fields])
    on_clause = " AND\n        ".join(on_conditions)

    return f"""
    MERGE INTO {target_table} AS {target_alias}
    USING {source_query} AS src
    ON
        {on_clause}
    WHEN MATCHED AND (
        {when_matched_condition}
    ) THEN
        UPDATE SET
            {update_clause}
    WHEN NOT MATCHED BY TARGET THEN
        INSERT (
            {insert_fields_clause}
        )
        VALUES (
            {insert_values_clause}
        );
    """


def generate_new_records_insert(
    sttm_df: pd.DataFrame, meta: Dict[str, str], config: PipelineConfig
) -> str:
    (
        tgt_fields,
        select_exprs,
        business_keys,
        compare_fields,
        base_table,
        join_clauses,
        source_fields_clause,
        source_alias_fields,
    ) = build_mapping_from_sttm(sttm_df, "late_arriving_insert", meta, config)

    target_table = meta["tgt_full_name"]
    target_alias = meta["tgt_alias"]
    on_clause = " AND\n        ".join(
        [f"{target_alias}.{k} <=> src.{k}" for k in business_keys]
    )

    compare_type_map = {
        str(row["Target Field Name"])
        .strip(): str(row["Target Data Type With Length"])
        .strip()
        for _, row in sttm_df.iterrows()
        if str(row["Target Field Name"]).strip() in compare_fields
    }

    compare_clause_parts = []
    for key, dtype in compare_type_map.items():
        dtype_l = dtype.lower()
        if dtype_l.startswith(("varchar", "char", "string")):
            compare_clause_parts.append(
                f"COALESCE({target_alias}.{key}, '') <> COALESCE(src.{key}, '')"
            )
        elif dtype_l.startswith(("double", "float", "decimal")):
            compare_clause_parts.append(
                f"COALESCE({target_alias}.{key}, 0.0) <> COALESCE(src.{key}, 0.0)"
            )
        elif dtype_l.startswith(("int", "bigint", "long", "short", "byte")):
            compare_clause_parts.append(
                f"COALESCE({target_alias}.{key}, 0) <> COALESCE(src.{key}, 0)"
            )
        elif dtype_l.startswith("date"):
            compare_clause_parts.append(
                f"COALESCE({target_alias}.{key}, CAST('1900-01-01' AS DATE)) <> "
                f"COALESCE(src.{key}, CAST('1900-01-01' AS DATE))"
            )
        elif dtype_l.startswith(("timestamp", "timestamp_ltz", "timestamp_ntz")):
            compare_clause_parts.append(
                f"COALESCE({target_alias}.{key}, CAST('1900-01-01 00:00:00' AS TIMESTAMP)) <> "
                f"COALESCE(src.{key}, CAST('1900-01-01 00:00:00' AS TIMESTAMP))"
            )
        elif dtype_l.startswith(("bool", "boolean")):
            compare_clause_parts.append(
                f"COALESCE({target_alias}.{key}, FALSE) <> COALESCE(src.{key}, FALSE)"
            )
        else:
            compare_clause_parts.append(f"{target_alias}.{key} <> src.{key}")

    compare_clause = (
        " OR\n            ".join(compare_clause_parts)
        if compare_clause_parts
        else "1 = 0"
    )

    filter_list = get_filter_list(sttm_df)
    filter_string = f"WHERE {' AND '.join(filter_list)}" if filter_list else ""
    join_str = "".join([f"\n        {j}" for j in join_clauses])

    if config.src_system.lower() in ("eph", "pph"):
        distinct_word = "DISTINCT"
    elif config.table_name.startswith("lnd"):
        distinct_word = ""
    else:
        distinct_word = "DISTINCT"

    select_fields_clause = ",\n            ".join(select_exprs)

    source_query = f"""
    (
        SELECT {distinct_word}
            {select_fields_clause}
        FROM {base_table}{join_str}
        {filter_string}
    )
    """.strip()

    if config.table_name.startswith("lnd") or config.src_system.lower() in (
        "pph",
        "eph",
    ):
        source_query = f"(SELECT * FROM {source_query} WHERE RN = 1)"

    if config.operation_type == "data_load_with_hipaa" and not target_table.endswith(
        f"hipaa_quarantine_{config.table_name}"
    ):
        source_query = (
            f"(SELECT * FROM {source_query} WHERE consent_dispense_active_flag = 1)"
        )

    if config.operation_type == "data_load_with_hipaa" and target_table.endswith(
        f"hipaa_quarantine_{config.table_name}"
    ):
        source_query = (
            f"(SELECT * FROM {source_query} WHERE consent_dispense_active_flag = 0)"
        )

    lookup_sql = control_lookup_sql(meta, config)
    tgt_fields_clause = ", ".join([f"src.{t}" for t in tgt_fields])
    insert_fields_clause = ", ".join(tgt_fields)

    return f"""
    INSERT INTO {target_table} (
        {insert_fields_clause}
    )
    SELECT
        {tgt_fields_clause}
    FROM {source_query} AS src
    LEFT JOIN {target_table} AS {target_alias}
    ON
        {on_clause}
    WHERE
        {target_alias}.lst_updt_ts BETWEEN ({lookup_sql}) AND CURRENT_TIMESTAMP()
        AND {target_alias}.is_actv = FALSE
        AND {target_alias}.etl_job_id = '{config.job_id}'
        AND (
            {compare_clause}
        );
    """


def infer_table_names_and_schema(
    sttm_df: pd.DataFrame, table_name: str
) -> Dict[str, str]:
    try:
        src_row = sttm_df[sttm_df["Source Table/File Name"].notnull()].iloc[0]
        tgt_row = sttm_df[sttm_df["Target Table/File Name"].notnull()].iloc[0]
    except Exception as e:
        raise ValueError(f"Could not infer table names/schema: {e}")

    return {
        "src_table": str(src_row["Source Table/File Name"]).strip(),
        "src_full_name": (
            f"{str(src_row['Source Catalog Name']).strip()}."
            f"{str(src_row['Source Schema Name']).strip()}."
            f"{str(src_row['Source Table/File Name']).strip()}"
        ),
        "src_alias": str(src_row["Source Table Alias"]).strip(),
        "tgt_schema": str(tgt_row["Target Table Schema"]).strip(),
        "tgt_table": str(tgt_row["Target Table/File Name"]).strip(),
        "tgt_full_name": (
            f"{str(tgt_row['Target Table Catalog']).strip()}."
            f"{str(tgt_row['Target Table Schema']).strip()}."
            f"{str(tgt_row['Target Table/File Name']).strip()}"
        ),
        "tgt_alias": str(tgt_row["Target Table Alias"]).strip(),
        "tgt_catalog": str(tgt_row["Target Table Catalog"]).strip(),
        "src_system": str(tgt_row["Source System"]).strip(),
        "dq_error_table": (
            f"{str(tgt_row['Target Table Catalog']).strip()}."
            f"{str(tgt_row['Target Table Schema']).strip()}."
            f"dq_error_{table_name}"
        ),
    }

In [0]:
COLUMN_NAME_MAP = {
    "source catalog name": "Source Catalog Name",
    "source schema name": "Source Schema Name",
    "source table/file name": "Source Table/File Name",
    "source table alias": "Source Table Alias",
    "source field name (table order)": "Source Field Name",
    "data type with length": "Data Type with length",
    "source business keys": "Source Business keys",
    "transformation": "Transformation",
    "joins": "Joins",
    "target table catalog": "Target Table Catalog",
    "target table schema": "Target Table Schema",
    "target table/file name": "Target Table/File Name",
    "target table alias": "Target Table Alias",
    "target field name": "Target Field Name",
    "target column description": "Target Column Description",
    "target data type with length": "Target Data Type With Length",
    "null / not null": "Null / Not Null",
    "direct vs indirect field mapping": "Direct vs Indirect Field Mapping",
    "primary key": "Primary Key",
    "foreign key": "Foreign Key",
    "source system": "Source System",
}


REQUIRED_COLUMNS = [
    "Source Catalog Name",
    "Source Schema Name",
    "Source Table/File Name",
    "Source Table Alias",
    "Source Field Name",
    "Joins",
    "Target Table Catalog",
    "Target Table Schema",
    "Target Table/File Name",
    "Target Table Alias",
    "Target Field Name",
    "Target Data Type With Length",
    "Direct vs Indirect Field Mapping",
    "Primary Key",
    "Foreign Key",
    "Transformation",
]


AUDIT_FIELD_DEFAULTS = {
    # 'etl_job_id': "NULL AS etl_job_id",
    "etl_src_id": "'LND' AS etl_src_id",
    "etl_src_rec_id": "NULL AS etl_src_rec_id",
    "src_dt": "NULL AS src_dt",
    "crtd_dt": "CURRENT_DATE() AS crtd_dt",
    "crtd_by": "'SYSTEM' AS crtd_by",
    "updt_dt": "CURRENT_DATE() AS updt_dt",
    "updt_by": "'SYSTEM' AS updt_by",
    "etl_dt": "CURRENT_TIMESTAMP() AS etl_dt",
    "eff_strt_dt": "CURRENT_DATE() AS eff_strt_dt",
    "eff_end_dt": "DATE '9999-12-31' AS eff_end_dt",
    "lst_updt_ts": "CURRENT_TIMESTAMP() AS  lst_updt_ts",
    "is_actv": "TRUE AS is_actv",
    "ext_fld_1": "NULL AS ext_fld_1",
    # opn_flg is handled separately
}

AUDIT_FIELD_DEFAULTS_COMPARE = {
    #'etl_job_id': "NULL AS etl_job_id",
    "etl_src_id": "'LND' AS etl_src_id",
    "etl_src_rec_id": "NULL AS etl_src_rec_id",
    "src_dt": "NULL AS src_dt",
    "crtd_dt": "CURRENT_DATE() AS crtd_dt",
    "crtd_by": "'SYSTEM' AS crtd_by",
    "updt_dt": "CURRENT_DATE() AS updt_dt",
    "updt_by": "'SYSTEM' AS updt_by",
    "etl_dt": "CURRENT_TIMESTAMP() AS etl_dt",
    "eff_strt_dt": "CURRENT_DATE() AS eff_strt_dt",
    "eff_end_dt": "DATE '9999-12-31' AS eff_end_dt",
    "lst_updt_ts": "CURRENT_TIMESTAMP() AS lst_updt_ts",
    "is_actv": "TRUE AS is_actv",
    "ext_fld_1": "NULL AS ext_fld_1",
    "opn_flg": "NULL AS opn_flg",
}

In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.
def main():
    """Main pipeline execution function."""
    config = load_config()
    dq_path = get_dq_path(config.volume_path)
    
    print(f"Using metadata table: {config.metadata_table}")
    print(f"Using DQ path: {dq_path}")
    
    meta = None
    ddl_sql = ""
    merge_sql = ""
    insert_sql = ""
    pipeline_start_ts = datetime.utcnow()
    
    try:
        # ===== PIPELINE START =====

        
        log_info(
            config,
            "PIPELINE_START",
            "Pipeline execution started",
            details={
                "table_name": config.table_name,
                "pipeline_name": config.pipeline_name,
                "job_id": config.job_id,
                "batch_id": getattr(config, 'batch_id', None)
            }
        )
        # ===== INITIALIZE CONTROL TABLE =====
        log_info(
            config,
            "INIT_CONTROL_TABLE",
            "Initializing job run in control table"
        )
        
        control_insert_sql = generate_control_table_insert(
            config.control_table,
            config.pipeline_name,
            meta["tgt_table"],
            meta["tgt_schema"],
            meta["src_system"],
            config.job_id,
            batch_id=getattr(config, 'batch_id', None)
        )
        
        print("--- Executing: Control Table Insert ----")
        print(control_insert_sql)
        exec_sql("Control Table Insert", control_insert_sql)
        
        log_success(
            config,
            "INIT_CONTROL_TABLE",
            "Job run initialized in control table"
        )

        
        
        # ===== LOAD METADATA =====
        log_info(
            config,
            "LOAD_METADATA",
            "Loading metadata from table",
            table_name=config.table_name,
        

        )
        
        metadata_df = load_metadata_from_table(config, COLUMN_NAME_MAP)
        metadata_row_count = len(metadata_df)
        
        sttm_ddl_df = metadata_df.copy()
        sttm_df = metadata_df[
            metadata_df["Transformation"].astype(str).str.strip() != "Autogenerated"
        ].copy()
        sttm_df = reorder_etl_job_id_last(sttm_df)
        sttm_df = sttm_df.fillna("")
        pipeline_sttm_df = sttm_df.copy()
        
        log_success(
            config,
            "LOAD_METADATA",
            f"Loaded {metadata_row_count} metadata records",
            rows_out=metadata_row_count,
            table_name=config.table_name,
            details={"filtered_rows": len(sttm_df)}
        )
        
        # ===== VALIDATE COLUMNS =====
        log_info(
            config,
            "VALIDATE_METADATA",
            "Validating required columns in metadata"
        )
        
        validate_required_columns(sttm_df)
        
        log_success(
            config,
            "VALIDATE_METADATA",
            "Metadata validation successful"
        )
        
        # ===== INFER TABLE NAMES =====
        log_info(
            config,
            "INFER_SCHEMA",
            "Inferring table names and schema from metadata"
        )
        
        meta = infer_table_names_and_schema(pipeline_sttm_df, config.table_name)
        
        log_success(
            config,
            "INFER_SCHEMA",
            "Schema inference completed",
            details={
                "tgt_table": meta.get("tgt_table"),
                "tgt_schema": meta.get("tgt_schema"),
                "src_system": meta.get("src_system")
            }
        )
        
        # ===== GENERATE DDL =====
        log_info(
            config,
            "GENERATE_DDL",
            "Generating DDL statements",
            table_name=f"{meta.get('tgt_schema')}.{meta.get('tgt_table')}"
        )
        
        ddl_sql, masking_commands = generate_ddl(sttm_ddl_df, "create_table")
        ddl_sql = ddl_sql.replace('"""', "'")
        
        log_success(
            config,
            "GENERATE_DDL",
            f"DDL generated with {len(masking_commands)} masking commands",
            details={"masking_commands_count": len(masking_commands)}
        )
        
        # ===== EXECUTE DDL (if create_table) =====
        if config.operation_type.lower() == "create_table":
            log_info(
                config,
                "EXECUTE_DDL",
                "Executing DDL to create table"
            )
            
            # Apply masking commands
            for i, masking_command in enumerate(masking_commands, start=1):
                log_info(
                    config,
                    "APPLY_MASKING",
                    f"Applying masking command {i}",
                    details={"command_number": i}
                )
                # exec_sql(f"Masking Command {i}", masking_command)
            
            if ddl_sql:
                print("DDL SQL:", ddl_sql)
                # exec_sql("Create Table", ddl_sql)
                log_success(
                    config,
                    "EXECUTE_DDL",
                    "Table created successfully",
                    table_name=f"{meta.get('tgt_schema')}.{meta.get('tgt_table')}"
                )
        
        
        
        # ===== GENERATE MERGE STATEMENT =====
        log_info(
            config,
            "GENERATE_MERGE",
            "Generating SCD2 MERGE statement"
        )
        
        merge_sql = generate_merge_statement_from_sttm(pipeline_sttm_df, meta, config)
        merge_sql = merge_sql.replace('"""', "'").replace('"', "'")
        
        log_success(
            config,
            "GENERATE_MERGE",
            "MERGE statement generated successfully"
        )
        
        # ===== EXECUTE MERGE =====
        log_info(
            config,
            "EXECUTE_MERGE",
            "Executing SCD2 MERGE statement"
        )
        
        print("----- Executing: SCD2 MERGE --- ")
        print(merge_sql)
        # exec_sql("SCD2 MERGE", merge_sql)
        
        log_success(
            config,
            "EXECUTE_MERGE",
            "SCD2 MERGE executed successfully"
        )
        
        # ===== GENERATE INSERT STATEMENT =====
        log_info(
            config,
            "GENERATE_INSERT",
            "Generating INSERT statement for new/late arriving records"
        )
        
        insert_sql = generate_new_records_insert(pipeline_sttm_df, meta, config)
        insert_sql = insert_sql.replace('"""', "'").replace('"', "'")
        
        log_success(
            config,
            "GENERATE_INSERT",
            "INSERT statement generated successfully"
        )
        
        # ===== EXECUTE INSERT =====
        log_info(
            config,
            "EXECUTE_INSERT",
            "Executing INSERT for new/late arriving records"
        )
        
        print("--- Executing: Insert New/Late Arriving Records ---")
        print(insert_sql)
        # exec_sql("Insert New/Late Arriving Records", insert_sql)
        
        log_success(
            config,
            "EXECUTE_INSERT",
            "INSERT for new/late arriving records executed successfully"
        )
        
        # ===== UPDATE CONTROL TABLE (SUCCESS) =====
        log_info(
            config,
            "UPDATE_CONTROL_TABLE",
            "Updating control table with success status"
        )
        
        control_table_update = generate_control_table_update(
            config=config,
            meta=meta,
            merge_query=merge_sql,
            insert_query=insert_sql,
            ddl_query=ddl_sql,
            status="SUCCESS",
            notes="Pipeline completed successfully"
        )
        
        print("--- Executing: Control Table Update (Success) ---")
        print(control_table_update)
        exec_sql("Control Table Update Success", control_table_update)
        
        log_success(
            config,
            "UPDATE_CONTROL_TABLE",
            "Control table updated with success status"
        )
        
        # ===== PIPELINE COMPLETE =====
        pipeline_end_ts = datetime.utcnow()
        total_duration = (pipeline_end_ts - pipeline_start_ts).total_seconds()
        log_success(
            config,
            "PIPELINE_COMPLETE",
            "Pipeline execution completed successfully",
            details={
                "table_name": config.table_name,
                "target_table": f"{meta.get('tgt_schema')}.{meta.get('tgt_table')}",
                "job_id": config.job_id
            }
        )
        
        print("========================================")
        print("✓ PIPELINE EXECUTION SUCCESSFUL")
        print("========================================")
        
    except Exception as e:
        import traceback
        
        error_msg = str(e)
        stack_trace = traceback.format_exc()
        
        print("========================================")
        print("✗ PIPELINE EXECUTION FAILED")
        print("========================================")
        print(f"Error: {error_msg}")
        print(f"Stack Trace:\n{stack_trace}")
        
        # Log the error
        log_error(
            config,
            "PIPELINE_FAILED",
            f"Pipeline execution failed: {error_msg}",
            error=e,
            details={
                "table_name": config.table_name,
                "job_id": config.job_id
            }
        )
        
        # Update control table with failure status
        if meta:
            try:
                log_info(
                    config,
                    "UPDATE_CONTROL_TABLE_FAILED",
                    "Updating control table with failure status"
                )
                
                control_update_failed = generate_control_table_update(
                    config=config,
                    meta=meta,
                    merge_query=merge_sql,
                    insert_query=insert_sql,
                    ddl_query=ddl_sql,
                    status="FAILED",
                    notes=f"Pipeline failed: {error_msg}"
                )
                
                exec_sql("Control Table Update Failed", control_update_failed)
                
                log_success(
                    config,
                    "UPDATE_CONTROL_TABLE_FAILED",
                    "Control table updated with failure status"
                )
                
            except Exception as control_err:
                control_error_msg = str(control_err)
                print(f"Failed to update control table: {control_error_msg}")
                
                log_error(
                    config,
                    "CONTROL_TABLE_UPDATE_FAILED",
                    f"Failed to update control table: {control_error_msg}",
                    error=control_err
                )
        else:
            # Meta not available, can't update control table properly
            print("Meta data not available, skipping control table update")
            log_warning(
                config,
                "CONTROL_TABLE_UPDATE_SKIPPED",
                "Meta data not available, control table update skipped",
                details={"reason": "meta_not_initialized"}
            )
        
        # Re-raise the original exception
        raise




In [0]:
if __name__ == '__main__':
    main()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-3984258203693638>, line 2
      1 if __name__ == '__main__':
----> 2     main()

NameError: name 'main' is not defined